# Bài toán 3: Phân tích Cảm xúc Văn bản (NLP) - Knowledge Distillation

Ở bài toán này, vì dữ liệu thô từ YouTube Comments không có nhãn (label) sẵn, ta sẽ áp dụng kỹ thuật **Teacher-Student (Knowledge Distillation)**:\n1. **Teacher**: Dùng mô hình BERT đa ngữ của Hugging Face để gán nhãn tự động (Pseudo-labeling) cho một tập dữ liệu nhỏ.\n2. **Student**: Train một mô hình TF-IDF + Logistic Regression siêu nhẹ dựa trên các nhãn đó để có thể dự đoán cực nhanh.\n3. **MLflow**: Theo dõi toàn bộ quá trình train Student, log siêu tham số, metrics và lưu mô hình Scikit-Learn.

In [1]:
!pip install -q duckdb transformers sentencepiece wordcloud mlflow==3.16.1 pysocks kaggle scikit-learn

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.0/50.0 kB 3.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 kB 4.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.3/44.3 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.7/11.7 MB 69.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.8/3.8 MB 41.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 57.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 268.7/268.7 kB 14.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 148.8/148.8 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 228.4/228.4 kB 13.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 136.5/136.5 kB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 148.7/148.7 kB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1

In [4]:
# Upload file colab_utils.py từ máy tính
from google.colab import files
uploaded = files.upload()
print("Uploaded:", list(uploaded.keys()))

import importlib
import colab_utils
importlib.invalidate_caches()
importlib.reload(colab_utils)
print("File đang dùng:", colab_utils.__file__)


Saving colab_utils.py to colab_utils.py
Uploaded: ['colab_utils.py']
File đang dùng: /content/colab_utils.py


In [6]:
import colab_utils
from google.colab import userdata

# colab_utils.install_tailscale()

status = colab_utils.setup_tailscale()
print("Connected check:", colab_utils.tailscale_is_connected())


Đang cài Tailscale...
Cài Tailscale thành công.
100.94.113.41    google-colab-3  taohshsh1303@  linux    -                           
100.116.34.36    ai-lab          taohshsh1303@  linux    -                           
100.118.178.119  baongoc         taohshsh1303@  windows  -                           
100.110.38.52    google-colab-1  taohshsh1303@  linux    offline, last seen 9h ago   
100.115.170.30   google-colab-2  taohshsh1303@  linux    offline, last seen 1m ago   
100.66.218.91    google-colab    taohshsh1303@  linux    offline, last seen 10h ago  
100.70.174.90    vmlab1          taohshsh1303@  linux    -
HTTP proxy: http://localhost:1056
SOCKS5 proxy: socks5://localhost:1055
Connected check: True


In [7]:
MLFLOW_URI = "http://100.116.34.36:8080"
EXPERIMENT_NAME = "NLP-Sentiment"

colab_utils.setup_mlflow(
    server_uri=MLFLOW_URI,
    experiment=EXPERIMENT_NAME,
)

MLflow server phản hồi HTTP 200: http://100.116.34.36:8080
MLflow tracking URI: http://100.116.34.36:8080
Experiment: NLP-Sentiment


'http://100.116.34.36:8080'

In [9]:

import duckdb
import pandas as pd
import numpy as np
import os

# Kiểm tra credentials đã được cấu hình
# MINIO_ACCESS_KEY = os.environ.get("MINIO_ACCESS_KEY", "minioadmin")
# MINIO_SECRET_KEY = os.environ.get("MINIO_SECRET_KEY", "minioadmin123")

# if not MINIO_ACCESS_KEY or not MINIO_SECRET_KEY:
#     raise ValueError(
#         "MINIO_ACCESS_KEY hoặc MINIO_SECRET_KEY đang rỗng. "
#         "Hãy lấy credentials từ cấu hình hiện có của bạn."
#     )


from google.colab import userdata

MINIO_ACCESS_KEY = userdata.get("MINIO_ACCESS_KEY")
MINIO_SECRET_KEY = userdata.get("MINIO_SECRET_KEY")

if not MINIO_ACCESS_KEY or not MINIO_SECRET_KEY:
    raise ValueError(
        "Thiếu MINIO_ACCESS_KEY hoặc MINIO_SECRET_KEY "
        "trong Colab Secrets."
    )


# Kết nối DuckDB
con = duckdb.connect(":memory:")
con.execute("INSTALL httpfs")
con.execute("LOAD httpfs")

# Cấu hình MinIO
con.execute(f"""
    CREATE OR REPLACE SECRET minio_secret (
        TYPE S3,
        KEY_ID '{MINIO_ACCESS_KEY.replace("'", "''")}',
        SECRET '{MINIO_SECRET_KEY.replace("'", "''")}',
        ENDPOINT '20.41.113.183:9000',
        URL_STYLE 'path',
        USE_SSL false
    )
""")

# Đọc dữ liệu
df_yt = con.execute("""
    SELECT text
    FROM read_parquet(
        's3://football-lake/dwh/mart_ml_text_youtube_comments.parquet'
    )
    WHERE text IS NOT NULL
    LIMIT 2000
""").df()

df_yt = df_yt.dropna(subset=["text"])
texts = df_yt["text"].astype(str).tolist()

print(f"Số lượng bình luận YouTube: {len(texts)}")
display(df_yt.head())



Số lượng bình luận YouTube: 2000


,text
0,Minh hoàng không thể thay thế được đình bắc vừ...
1,"Chelsea thất thủ thật đáng tiếc, còn trận đấu ..."
2,sao vẫn có nhiều thằng mơ mộng đụng hàng sao c...
3,"Tôi nghĩ FA có thể đấm thẳng tay, bọn UAE cũng..."
4,Estupinian is always in the wrong position whe...


In [10]:

# ==========================================
# 1. TEACHER: TẠO PSEUDO-LABELS BẰNG BERT (GPU)
# ==========================================
import torch
from transformers import pipeline
from tqdm.auto import tqdm
import pandas as pd
import time

# Chọn GPU nếu có, nếu không thì dùng CPU
device = 0 if torch.cuda.is_available() else -1

if device == 0:
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("Không có GPU, đang chạy bằng CPU")

# Tải Teacher Model
print("Đang tải Teacher Model (BERT)...")
teacher = pipeline(
    "sentiment-analysis",
    model="nlptown/bert-base-multilingual-uncased-sentiment",
    device=device
)

# Gán nhãn tự động
print("Đang gán nhãn pseudo-labels...")
start = time.time()

results = teacher(
    texts,
    batch_size=32 if device == 0 else 8,
    truncation=True,
    max_length=512
)

# Chuyển 5 mức sao thành 3 lớp
pseudo_labels = []

for res in results:
    stars = int(res["label"][0])

    if stars <= 2:
        pseudo_labels.append(0)  # Negative
    elif stars == 3:
        pseudo_labels.append(1)  # Neutral
    else:
        pseudo_labels.append(2)  # Positive

df_yt["label"] = pseudo_labels

print(f"Hoàn thành sau {time.time() - start:.2f} giây")
print("Phân phối nhãn:")
print(df_yt["label"].value_counts().sort_index())


GPU: Tesla T4
Đang tải Teacher Model (BERT)...


config.json:   0%|          | 0.00/953 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  669MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/39.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/872k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

Đang gán nhãn pseudo-labels...
Hoàn thành sau 12.60 giây
Phân phối nhãn:
label
0    1042
1     374
2     584
Name: count, dtype: int64


In [11]:

# ==========================================
# 2. STUDENT: TF-IDF + LOGISTIC REGRESSION + MLFLOW
# ==========================================
import mlflow
import mlflow.sklearn
import json

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    f1_score
)

# Chuẩn bị dữ liệu
data = df_yt.dropna(subset=["text", "label"]).copy()
data["text"] = data["text"].astype(str)

X_train, X_test, y_train, y_test = train_test_split(
    data["text"],
    data["label"].astype(int),
    test_size=0.2,
    random_state=42,
    stratify=data["label"]
)

# Siêu tham số
params = {
    "tfidf__max_features": 5000,
    "tfidf__ngram_range": "(1, 2)",
    "lr__C": 1.0,
    "lr__max_iter": 500
}

pipeline_model = Pipeline([
    ("tfidf", TfidfVectorizer(
        max_features=params["tfidf__max_features"],
        ngram_range=(1, 2)
    )),
    ("lr", LogisticRegression(
        C=params["lr__C"],
        max_iter=params["lr__max_iter"]
    ))
])

with mlflow.start_run(run_name="student_tfidf_lr"):

    # Log tham số
    mlflow.log_params(params)
    mlflow.log_param("train_size", len(X_train))
    mlflow.log_param("test_size", len(X_test))
    mlflow.log_param("model_type", "TF-IDF + Logistic Regression")

    # Huấn luyện
    print("Đang huấn luyện Student Model...")
    pipeline_model.fit(X_train, y_train)

    # Đánh giá
    preds = pipeline_model.predict(X_test)

    acc = accuracy_score(y_test, preds)
    macro_f1 = f1_score(
        y_test, preds, average="macro", zero_division=0
    )
    weighted_f1 = f1_score(
        y_test, preds, average="weighted", zero_division=0
    )

    labels = [0, 1, 2]
    target_names = ["Negative", "Neutral", "Positive"]

    report = classification_report(
        y_test,
        preds,
        labels=labels,
        target_names=target_names,
        output_dict=True,
        zero_division=0
    )

    print(f"Accuracy: {acc:.4f}")
    print(f"Macro F1: {macro_f1:.4f}")
    print(f"Weighted F1: {weighted_f1:.4f}")
    print("\nClassification Report:")
    print(classification_report(
        y_test,
        preds,
        labels=labels,
        target_names=target_names,
        zero_division=0
    ))

    # Log metrics
    mlflow.log_metric("test_accuracy", acc)
    mlflow.log_metric("test_macro_f1", macro_f1)
    mlflow.log_metric("test_weighted_f1", weighted_f1)

    for class_name in target_names:
        mlflow.log_metric(
            f"{class_name.lower()}_f1",
            report[class_name]["f1-score"]
        )

    # Lưu báo cáo đánh giá thành artifact
    with open("/tmp/classification_report.json", "w") as f:
        json.dump(report, f, indent=2)

    mlflow.log_artifact("/tmp/classification_report.json")

    # Lưu mô hình
    model_info = mlflow.sklearn.log_model(
        pipeline_model,
        name="student_nlp_model"
    )

    print(f"\nModel URI: {model_info.model_uri}")

student_uri = model_info.model_uri


Đang huấn luyện Student Model...
Accuracy: 0.5900
Macro F1: 0.4518
Weighted F1: 0.5422

Classification Report:
              precision    recall  f1-score   support

    Negative       0.61      0.86      0.71       208
     Neutral       0.41      0.09      0.15        75
    Positive       0.57      0.44      0.49       117

    accuracy                           0.59       400
   macro avg       0.53      0.46      0.45       400
weighted avg       0.56      0.59      0.54       400




Model URI: models:/m-efab628c740e46ca8bd90fe96a253fe4
🏃 View run student_tfidf_lr at: http://100.116.34.36:8080/#/experiments/3/runs/dd8e405adbc0454ca5b006f681fe1127
🧪 View experiment at: http://100.116.34.36:8080/#/experiments/3


In [12]:

# ==========================================
# 3. LOAD MODEL TỪ MLFLOW VÀ DỰ ĐOÁN
# ==========================================
import mlflow.sklearn

print("Đang tải lại Student Model từ MLflow...")

loaded_nlp_model = mlflow.sklearn.load_model(student_uri)

sample_texts = [
    "Cầu thủ này đá quá tệ, thật thất vọng!",
    "Trận đấu diễn ra bình thường, không có gì nổi bật.",
    "Bàn thắng tuyệt vời! Cảm xúc vỡ òa, quá đỉnh!",
    "HLV chiến thuật quá sai lầm, đội bóng mất định hướng."
]

predictions = loaded_nlp_model.predict(sample_texts)

label_map = {
    0: "Tiêu cực 😡",
    1: "Trung tính 😐",
    2: "Tích cực 😍"
}

print("\nKẾT QUẢ DỰ ĐOÁN:")

for text, pred in zip(sample_texts, predictions):
    label = label_map.get(int(pred), "Không xác định")
    print(f"Text: {text}\n => Phân loại: {label}\n")


Đang tải lại Student Model từ MLflow...


KeyboardInterrupt: 